# Project 2 — Traffic Lens 🚗

Detect → track → count vehicles, and meet the 2026 tooling: **YOLO26** (NMS-free),
**`supervision`** (tracking/annotation), and **open-vocabulary auto-labeling**. This is
your YOLOv5 knowledge, upgraded.

In [ ]:
import supervision as sv
from ultralytics import YOLO
print("supervision", sv.__version__)

## 1. YOLO26 in one line — what changed since v5
Anchors gone (v8), NMS gone (v10/26 train end-to-end with one-to-one matching). The API
you remember still works. First call downloads weights (~6 MB for nano).

In [ ]:
model = YOLO("yolo26n.pt")   # falls back automatically if your ultralytics predates yolo26
print("classes model knows:", len(model.names), "COCO classes")
print("vehicles we care about:", {k: v for k, v in model.names.items() if k in (2,3,5,7)})

In [ ]:
# Detect on Ultralytics' sample bus image
result = model("https://ultralytics.com/images/bus.jpg", verbose=False)[0]
det = sv.Detections.from_ultralytics(result)
print(f"{len(det)} objects detected; classes: {[model.names[c] for c in det.class_id]}")

## 2. Annotate with supervision
`supervision` is the workhorse: annotators, trackers, zones, line counters. Learn it once,
use it in Projects 2 AND 3.

In [ ]:
import cv2, numpy as np, urllib.request
from PIL import Image
arr = np.array(Image.open(urllib.request.urlopen("https://ultralytics.com/images/bus.jpg")).convert("RGB"))
box = sv.BoxAnnotator(); lab = sv.LabelAnnotator()
labels = [f"{model.names[c]} {conf:.2f}" for c, conf in zip(det.class_id, det.confidence)]
annotated = lab.annotate(box.annotate(arr.copy(), det), det, labels=labels)
plt = __import__("matplotlib.pyplot", fromlist=["x"]); plt.figure(figsize=(5,7))
plt.imshow(annotated); plt.axis("off"); plt.title("YOLO26 + supervision"); plt.show()

## 3. Tracking + counting on video
`ByteTrack` assigns persistent IDs across frames; `LineZone` counts crossings. Full pipeline
(with speed via homography) lives in `track_count_speed.py`.

In [ ]:
# from supervision.assets import VideoAssets, download_assets
# video = download_assets(VideoAssets.VEHICLES)      # uncomment to fetch the demo clip
# Then run in a terminal:  python track_count_speed.py --max-frames 100
# -> outputs/annotated.mp4  (open it: traces, IDs, live in/out counts)
print("Run `python track_count_speed.py --max-frames 100` and open outputs/annotated.mp4")

## 4. Auto-labeling — the 2026 data workflow
Grounding DINO detects from a **text prompt** ("car. truck. bus.") — classes it was never
explicitly trained on. You *correct* its output instead of labeling from scratch (~5× faster).
`autolabel.py` writes YOLO-format labels + a preview.

⚠️ **The honest catch (your blog angle):** the model silently misses hard cases — the exact
ones you most need labeled. Never ship auto-labels unreviewed.

In [ ]:
# python autolabel.py --source my_frames/ --classes "car,truck,bus,motorcycle"
# -> autolabel_dataset/{images,labels,preview,data.yaml}
print("Auto-label workflow: autolabel.py -> review in CVAT -> yolo train ...")

## 5. Benchmark on YOUR machine
`benchmark.py` times YOLO26 vs YOLO11 (add `--include-rtdetr` for the transformer detector).
The resulting table goes in your README — latency numbers are CV-engineer currency.

## 🔨 Your turn (see README for the full list)
1. Record 10 min of real traffic on your phone; extract frames; auto-label; correct ~300.
2. Time yourself labeling from scratch vs correcting — that ratio headlines blog #2.
3. Fine-tune YOLO26n on your data (⚡ Kaggle GPU); compare mAP to pretrained.
4. Calibrate speed with lane-width geometry.

➡️ **Next:** `project3_soccer_analytics/notebook_p3_soccer.ipynb`